# 🔥 Level 3 (Advanced) — High-Quality Face Swap + Identity Generation

Notebook ketiga **Learn-faceswap**. Ini versi *advanced* yang hasilnya jauh lebih bagus dari Notebook 2 (klasik). Berisi **2 mode**:

- **🅰️ MODE A — High-Quality Face Swap**: badan/background TETAP, hanya wajah diganti. Pakai **InsightFace `inswapper_128`** (swap berbasis AI) + **GFPGAN** (pertajam) + upscale.
- **🅱️ MODE B — Prompt-driven Identity Generation**: kasih 1 foto wajah + tulis *prompt*, AI bikin gambar BARU dengan identitas itu. Pakai **InstantID** di atas **Stable Diffusion XL**.

---
## ⚠️ WAJIB DIBACA DULU
1. **Aktifkan GPU**: menu `Runtime → Change runtime type → T4 GPU`. Tanpa GPU akan sangat lambat / gagal.
2. **Jalankan SATU mode dalam satu sesi.** Mode A & Mode B punya kebutuhan library berbeda. Setelah selesai Mode A, sebaiknya `Runtime → Restart session` sebelum mulai Mode B.
3. **Etika & legal**: gunakan **foto milikmu sendiri / yang kamu punya izinnya**. Jangan dipakai untuk meniru identitas orang lain secara menyesatkan (deepfake non-konsensual). Banyak negara sudah mengaturnya.
4. Beberapa model diunduh dari internet (ukuran besar, beberapa GB). URL/versi bisa berubah seiring waktu — kalau ada link mati, catatan di tiap cell memberi tahu cara menggantinya.

---

# 🅰️ MODE A — High-Quality Face Swap

Hasil: wajah orang dari **FACE** dipasang ke wajah di **BASE**, badan & background BASE tetap utuh, lalu dipertajam.

### Alur: deteksi+align → swap (inswapper) → restorasi (GFPGAN) → upscale

In [ ]:
# === MODE A | Langkah 1: install + unduh model swap ===
!pip install -q insightface onnxruntime-gpu opencv-python matplotlib

import os, urllib.request
os.makedirs('models', exist_ok=True)

# Helper unduh yang mengirim User-Agent (beberapa server menolak request tanpa UA).
def download(url, path):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=120) as r, open(path, 'wb') as f:
        f.write(r.read())
    return path

# Model 'inswapper_128.onnx'. Jika link utama mati, otomatis coba mirror kedua.
INSWAPPER = 'models/inswapper_128.onnx'
if not os.path.exists(INSWAPPER):
    mirrors = [
        'https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx',
        'https://huggingface.co/datasets/Gourieff/ReActor/resolve/main/models/inswapper_128.onnx',
    ]
    for m in mirrors:
        try:
            print('Mengunduh inswapper_128.onnx dari', m.split('/')[2], '...')
            download(m, INSWAPPER); break
        except Exception as e:
            print('  gagal:', e)
print('Model swap siap:', os.path.exists(INSWAPPER))

In [ ]:
# === MODE A | Langkah 2: load detektor + swapper ===
import cv2, numpy as np, insightface
from insightface.app import FaceAnalysis
from matplotlib import pyplot as plt

providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
app = FaceAnalysis(name='buffalo_l', providers=providers)
app.prepare(ctx_id=0, det_size=(640, 640))
swapper = insightface.model_zoo.get_model(INSWAPPER, providers=providers)

def show(img_bgr, title='', size=(7, 7)):
    plt.figure(figsize=size)
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title); plt.axis('off'); plt.show()

print('InsightFace (buffalo_l) + swapper siap.')

In [ ]:
# === MODE A | Langkah 3: siapkan gambar BASE + FACE ===
# BASE = gambar tujuan (badan/scene). FACE = wajah sumber yang ingin dipasang.
# Contoh memakai wajah AI (orang TIDAK nyata) dari thispersondoesnotexist.com
# -> 1024px, tajam, dan bebas masalah izin/etika. Tiap unduhan = wajah berbeda.
base_path, face_path = 'base.jpg', 'face.jpg'
try:
    download('https://thispersondoesnotexist.com/', base_path)
    download('https://thispersondoesnotexist.com/', face_path)
    print('Berhasil mengunduh 2 wajah contoh (AI).')
except Exception as e:
    print('Gagal unduh contoh:', e)
    print('>> Pakai gambar sendiri lewat Opsi upload di bawah.')

# --- Pakai foto sendiri? Hapus tanda # di bawah (disarankan FACE tajam & frontal) ---
# from google.colab import files
# print('Upload BASE:');  base_path = list(files.upload().keys())[0]
# print('Upload FACE:');  face_path = list(files.upload().keys())[0]

img_base = cv2.imread(base_path)
img_face = cv2.imread(face_path)
assert img_base is not None and img_face is not None, 'Gambar gagal dibaca. Cek unduhan / upload.'
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].imshow(cv2.cvtColor(img_base, cv2.COLOR_BGR2RGB)); ax[0].set_title('BASE (tujuan)'); ax[0].axis('off')
ax[1].imshow(cv2.cvtColor(img_face, cv2.COLOR_BGR2RGB)); ax[1].set_title('FACE (wajah sumber)'); ax[1].axis('off')
plt.show()

In [ ]:
# === MODE A | Langkah 4: lakukan SWAP ===
def swap_all_faces(base_bgr, face_bgr):
    src_faces = app.get(face_bgr)
    if not src_faces:
        raise ValueError('Wajah sumber (FACE) tidak terdeteksi. Pakai foto wajah yang lebih jelas.')
    # ambil wajah terbesar sebagai sumber identitas
    src = sorted(src_faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))[-1]
    targets = app.get(base_bgr)
    if not targets:
        raise ValueError('Wajah di gambar BASE tidak terdeteksi.')
    out = base_bgr.copy()
    for t in targets:                       # ganti semua wajah di BASE
        out = swapper.get(out, t, src, paste_back=True)
    return out

swapped = swap_all_faces(img_base, img_face)
show(swapped, 'HASIL SWAP (mentah, sebelum dipertajam)')

In [ ]:
# === MODE A | Langkah 5 (opsional tapi disarankan): pertajam wajah + upscale 2x ===
!pip install -q gfpgan basicsr facexlib realesrgan

# Perbaikan kompatibilitas: basicsr memakai modul torchvision yang sudah dihapus
import os, basicsr
deg = os.path.join(os.path.dirname(basicsr.__file__), 'data', 'degradations.py')
src = open(deg).read().replace('torchvision.transforms.functional_tensor',
                               'torchvision.transforms.functional')
open(deg, 'w').write(src)

from gfpgan import GFPGANer
restorer = GFPGANer(
    model_path='https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth',
    upscale=2, arch='clean', channel_multiplier=2, bg_upsampler=None)

_, _, enhanced = restorer.enhance(swapped, has_aligned=False, only_center_face=False, paste_back=True)
show(enhanced, 'HASIL SWAP + GFPGAN (lebih tajam, resolusi 2x)')
cv2.imwrite('hasil_swap_mode_a.png', enhanced)
print('Tersimpan: hasil_swap_mode_a.png')

In [ ]:
# === MODE A | Langkah 6: bandingkan ===
fig, ax = plt.subplots(1, 3, figsize=(16, 6))
for a, im, t in zip(ax, [img_base, img_face, enhanced], ['BASE asli', 'FACE sumber', 'HASIL akhir']):
    a.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); a.set_title(t); a.axis('off')
plt.show()

---
# 🅱️ MODE B — Prompt-driven Identity Generation (InstantID)

> 🔁 **Disarankan `Runtime → Restart session` dulu** sebelum menjalankan Mode B (supaya library bersih), lalu jalankan cell-cell di bawah dari sini.

Konsep: kamu kasih **1 foto wajah** + **prompt teks**, InstantID membuat **gambar baru** sesuai prompt namun **identitas wajah dipertahankan**. Bisa atur gaya, seed (variasi), dan kekuatan identitas.

InstantID memakai InsightFace untuk deteksi & ekstrak embedding wajah, lalu mengendalikan Stable Diffusion XL lewat IP-Adapter + ControlNet.

In [ ]:
# === MODE B | Langkah 1: install (versi di-pin agar pipeline InstantID kompatibel) ===
!pip install -q diffusers==0.27.2 transformers==4.39.3 accelerate==0.29.2 huggingface_hub==0.22.2 insightface==0.7.3 onnxruntime-gpu opencv-python einops
print('Jika ada error versi, sesuaikan: InstantID stabil di sekitar diffusers 0.25-0.27.')

In [ ]:
# === MODE B | Langkah 2: unduh pipeline + bobot InstantID + model wajah ===
import os, urllib.request

# (a) file pipeline kustom InstantID. Jika URL berubah, cari file
#     'pipeline_stable_diffusion_xl_instantid.py' di repo GitHub InstantID.
pipe_url = 'https://raw.githubusercontent.com/instantX-research/InstantID/main/pipeline_stable_diffusion_xl_instantid.py'
try:
    urllib.request.urlretrieve(pipe_url, 'pipeline_stable_diffusion_xl_instantid.py')
except Exception as e:
    urllib.request.urlretrieve('https://raw.githubusercontent.com/InstantID/InstantID/main/pipeline_stable_diffusion_xl_instantid.py',
                               'pipeline_stable_diffusion_xl_instantid.py')

# (b) bobot ControlNet + IP-Adapter dari HuggingFace
from huggingface_hub import hf_hub_download
hf_hub_download('InstantX/InstantID', 'ControlNetModel/config.json', local_dir='./checkpoints')
hf_hub_download('InstantX/InstantID', 'ControlNetModel/diffusion_pytorch_model.safetensors', local_dir='./checkpoints')
hf_hub_download('InstantX/InstantID', 'ip-adapter.bin', local_dir='./checkpoints')

# (c) model wajah 'antelopev2' untuk InsightFace
if not os.path.exists('models/antelopev2'):
    !wget -q https://github.com/deepinsight/insightface/releases/download/v0.7/antelopev2.zip -O antelopev2.zip
    !mkdir -p models && unzip -q -o antelopev2.zip -d models/
print('Aset InstantID siap. Isi models/antelopev2:', os.listdir('models/antelopev2') if os.path.exists('models/antelopev2') else 'TIDAK ADA')

In [ ]:
# === MODE B | Langkah 3: load pipeline InstantID + SDXL ===
import torch, cv2, numpy as np
from PIL import Image
from diffusers.models import ControlNetModel
from insightface.app import FaceAnalysis
from pipeline_stable_diffusion_xl_instantid import StableDiffusionXLInstantIDPipeline, draw_kps

app = FaceAnalysis(name='antelopev2', root='./', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

controlnet = ControlNetModel.from_pretrained('./checkpoints/ControlNetModel', torch_dtype=torch.float16)

# Base model SDXL. 'wangqixun/YamerMIX_v8' = pilihan demo InstantID (hasil bagus).
# Alternatif resmi: 'stabilityai/stable-diffusion-xl-base-1.0'
pipe = StableDiffusionXLInstantIDPipeline.from_pretrained(
    'wangqixun/YamerMIX_v8', controlnet=controlnet, torch_dtype=torch.float16)
pipe.load_ip_adapter_instantid('./checkpoints/ip-adapter.bin')
pipe.enable_model_cpu_offload()   # hemat VRAM, cocok untuk T4 16GB
print('Pipeline InstantID siap.')

In [ ]:
# === MODE B | Langkah 4: siapkan wajah referensi ===
# Default pakai 'face.jpg'. Untuk hasil terbaik: wajah TAJAM, TERANG, FRONTAL, ukuran besar.
import urllib.request
def _dl(url, path):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=120) as r, open(path, 'wb') as f:
        f.write(r.read())

face_ref_path = 'face.jpg'
if not os.path.exists(face_ref_path):
    # wajah AI (orang tidak nyata) sebagai contoh; ganti dengan fotomu sendiri di bawah
    _dl('https://thispersondoesnotexist.com/', face_ref_path)

# from google.colab import files
# face_ref_path = list(files.upload().keys())[0]

face_image = Image.open(face_ref_path).convert('RGB')
info = app.get(cv2.cvtColor(np.array(face_image), cv2.COLOR_RGB2BGR))
if not info:
    raise ValueError('Wajah tidak terdeteksi. Pakai foto wajah yang lebih jelas.')
info = sorted(info, key=lambda x: (x['bbox'][2]-x['bbox'][0])*(x['bbox'][3]-x['bbox'][1]))[-1]
face_emb = info['embedding']
face_kps = draw_kps(face_image, info['kps'])
print('Embedding wajah siap. Dimensi:', face_emb.shape)

In [ ]:
# === MODE B | Langkah 5: GENERATE gambar baru dari prompt ===
from IPython.display import display

prompt = 'cinematic photo of a person as an astronaut inside a spaceship, detailed space suit, dramatic lighting, ultra realistic, 8k'
negative = 'blurry, low quality, distorted, deformed, extra fingers, cartoon, painting, text, watermark'

image = pipe(
    prompt=prompt,
    negative_prompt=negative,
    image_embeds=face_emb,
    image=face_kps,
    controlnet_conditioning_scale=0.8,   # 0-1: seberapa ikut struktur/pose wajah referensi
    ip_adapter_scale=0.8,                # 0-1: seberapa kuat identitas dipertahankan
    num_inference_steps=30,
    guidance_scale=5.0,
    generator=torch.Generator(device='cpu').manual_seed(42)  # ganti angka seed -> variasi baru
).images[0]

image.save('hasil_instantid_mode_b.png')
print('Tersimpan: hasil_instantid_mode_b.png')
display(image)

## 🎛️ Cara mengatur hasil (Mode B)

| Parameter | Fungsi | Tips |
|-----------|--------|------|
| `prompt` | Deskripsi gambar yang diinginkan | Makin detail makin terarah |
| `seed` (`manual_seed`) | Angka acak awal | Seed sama = hasil sama; ganti angka = variasi baru |
| `ip_adapter_scale` | Kekuatan identitas wajah | Naikkan (0.8-1.0) bila wajah kurang mirip |
| `controlnet_conditioning_scale` | Ikut pose/struktur wajah referensi | Turunkan bila ingin pose lebih bebas |
| `num_inference_steps` | Jumlah langkah denoising | 30 cukup; lebih banyak = lebih halus tapi lambat |
| `guidance_scale` | Seberapa patuh ke prompt | 4-7 ideal untuk SDXL |

**Resolusi:** SDXL default ~1024x1024. Untuk lebih tinggi, jalankan hasilnya lewat upscaler (mis. Real-ESRGAN seperti di Mode A).

---
## 🧠 Ringkasan & Perbandingan

| | Mode A (Swap) | Mode B (InstantID) |
|---|---|---|
| Input | BASE + FACE | FACE + prompt teks |
| Yang berubah | Hanya wajah | Seluruh gambar (baru) |
| Background/badan | Tetap utuh | Dibuat ulang sesuai prompt |
| Kontrol kreatif | Rendah | Tinggi (lewat prompt) |
| VRAM | ~4-6 GB | ~12-15 GB |
| Colab gratis (T4) | Lancar | Bisa (pakai cpu offload) |

### 🔧 Tool/alternatif terbaik lain (open-source)
- **FaceFusion** — paling turnkey untuk swap foto & **video**, sudah ada enhancer built-in
- **IP-Adapter FaceID / PhotoMaker** — alternatif Mode B, bisa pakai beberapa foto wajah
- **CodeFormer** — alternatif GFPGAN untuk restorasi wajah
- **Real-ESRGAN** — upscaler umum untuk menaikkan resolusi keseluruhan gambar

### 💡 Kunci hasil bagus (rangkuman)
1. Foto wajah referensi **tajam, terang, frontal, besar di frame**
2. Selalu **pertajam** (GFPGAN/CodeFormer) setelah swap
3. **Upscale** untuk resolusi tinggi
4. Untuk Mode B: eksperimen `seed`, `ip_adapter_scale`, dan prompt

### ⚠️ Pengingat etika
Gunakan hanya foto milikmu / yang berizin. Jangan untuk meniru orang lain secara menyesatkan.